In [ ]:
from torch.optim.lr_scheduler import StepLR, ReduceLROnPlateau
"""
可以结合多个学习率调整策略，例如先使用 StepLR,
然后切换到 ReduceLROnPlateau。


"""


# 定义 StepLR 调整器, 每隔30个epoch，将学习率乘以0.1
step_scheduler = StepLR(optimizer, step_size=30, gamma=0.1) 

# 定义 ReduceLROnPlateau 调整器 ==》 当验证集损失连续10个epoch不下降时，将学习率乘以0.1
plateau_scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=10)

# 在训练过程中更新学习率
for epoch in range(num_epochs):
    for inputs, targets in train_loader:
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward() # 计算梯度
        optimizer.step() # 更新参数

    # 验证模型并获取验证集损失
    val_loss = validate(model, val_loader)

    # 更新学习率
    if epoch < 50:  # 在前50个epoch内使用 StepLR
        step_scheduler.step() 
    else: # 在第50个epoch开始使用 ReduceLROnPlateau
        plateau_scheduler.step(val_loss)

In [ ]:
# ------------------------------------------------------------- 使用梯度裁剪 -------------------------------------------------------------
""" 梯度裁剪可以防止梯度爆炸问题，特别是在训练深度网络时。"""
for inputs, targets in train_loader:
    optimizer.zero_grad()
    outputs = model(inputs)
    loss = criterion(outputs, targets)
    loss.backward()

    # 梯度裁剪
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

    optimizer.step()


